In [1]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset

dataset = load_dataset("go_emotions")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [3]:
import pandas as pd

df = dataset["train"].to_pandas()

print(df.head())

                                                text labels       id
0  My favourite food is anything I didn't have to...   [27]  eebbqej
1  Now if he does off himself, everyone will thin...   [27]  ed00q6i
2                     WHY THE FUCK IS BAYLESS ISOING    [2]  eezlygj
3                        To make her feel threatened   [14]  ed7ypvh
4                             Dirty Southern Wankers    [3]  ed0bdzj


In [4]:
df = df[df["labels"].map(len) == 1]
df["label"] = df["labels"].apply(lambda x: x[0])

In [5]:
label_names = dataset["train"].features["labels"].feature.names
df["emotion"] = df["label"].apply(lambda x: label_names[x])

In [6]:
target_emotions = ["anger", "joy", "sadness", "fear", "disgust", "neutral"]

df = df[df["emotion"].isin(target_emotions)]

In [7]:
df["emotion"] = df["emotion"].replace({
    "anger": "angry",
    "joy": "happy",
    "sadness": "sad"
})

In [8]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
# Encode labels
df["emotion_encoded"] = encoder.fit_transform(df["emotion"])

#BALANCE DATASET HERE
min_samples = df["emotion"].value_counts().min()

df_balanced = (
    df.groupby("emotion")
      .apply(lambda x: x.sample(min_samples, random_state=42))
      .reset_index(drop=True)
)

print(df_balanced["emotion"].value_counts())

emotion
angry      430
disgust    430
fear       430
happy      430
neutral    430
sad        430
Name: count, dtype: int64


C:\Users\pasha\AppData\Local\Temp\ipykernel_16004\2544852613.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min_samples, random_state=42))


In [9]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_balanced["text"],
    df_balanced["emotion_encoded"],
    test_size=0.2,
    stratify=df_balanced["emotion_encoded"],
    random_state=42
)

In [10]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(encoder.classes_)
)

C:\Users\pasha\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
W0225 15:12:28.542000 16004 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=160
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=160
)

In [13]:
import torch

class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmotionDataset(train_encodings, train_labels)
test_dataset = EmotionDataset(test_encodings, test_labels)

In [14]:
pip install --upgrade accelerate

  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.27.2
    Uninstalling accelerate-0.27.2:
      Successfully uninstalled accelerate-0.27.2
Note: you may need to restart the kernel to use updated packages.


In [15]:
pip install transformers==4.38.2

Note: you may need to restart the kernel to use updated packages.


In [16]:
pip uninstall transformers accelerate -y

Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Note: you may need to restart the kernel to use updated packages.


In [17]:
pip install transformers==4.38.2 accelerate==0.27.2 torch

  Using cached transformers-4.38.2-py3-none-any.whl.metadata (130 kB)
  Using cached accelerate-0.27.2-py3-none-any.whl.metadata (18 kB)
Using cached transformers-4.38.2-py3-none-any.whl (8.5 MB)
Using cached accelerate-0.27.2-py3-none-any.whl (279 kB)

   ---------------------------------------- 0/2 [accelerate]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transf

In [18]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")

    return {
        "accuracy": acc,
        "macro_f1": f1
    }

In [19]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",

    num_train_epochs=6,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    logging_dir="./logs",
    load_best_model_at_end=True
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\pasha\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.566498,0.513566,0.480211
2,No log,0.832656,0.734496,0.731482
3,No log,0.775899,0.742248,0.742994
4,0.938400,0.800664,0.753876,0.753034
5,0.938400,0.866591,0.746124,0.743290
6,0.938400,0.892400,0.748062,0.746486


Checkpoint destination directory ./results\checkpoint-129 already exists and is non-empty. Saving will proceed but saved results may be invalid.
C:\Users\pasha\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Checkpoint destination directory ./results\checkpoint-258 already exists and is non-empty. Saving will proceed but saved results may be invalid.
C:\Users\pasha\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Checkpoint destination directory ./results\checkpoint-387 already exists and is non-empty. Saving will proceed but saved results may be invalid.
C:\Users\pasha\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument

TrainOutput(global_step=774, training_loss=0.6780726459907315, metrics={'train_runtime': 1179.5869, 'train_samples_per_second': 10.499, 'train_steps_per_second': 0.656, 'total_flos': 166622755732992.0, 'train_loss': 0.6780726459907315, 'epoch': 6.0})

In [20]:
trainer.evaluate()

C:\Users\pasha\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.7758988738059998,
 'eval_accuracy': 0.7422480620155039,
 'eval_macro_f1': 0.742994127894144,
 'eval_runtime': 12.408,
 'eval_samples_per_second': 41.586,
 'eval_steps_per_second': 2.66,
 'epoch': 6.0}

In [21]:
import numpy as np
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred, target_names=encoder.classes_))

C:\Users\pasha\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

       angry       0.73      0.59      0.65        86
     disgust       0.63      0.67      0.65        86
        fear       0.75      0.87      0.81        86
       happy       0.94      0.85      0.89        86
     neutral       0.62      0.70      0.66        86
         sad       0.84      0.77      0.80        86

    accuracy                           0.74       516
   macro avg       0.75      0.74      0.74       516
weighted avg       0.75      0.74      0.74       516



In [22]:
model.save_pretrained("text_emotion_model")
tokenizer.save_pretrained("text_emotion_model")

import pickle
with open("text_label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

In [25]:
import torch
import numpy as np

def predict_text_emotion(text):
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=160
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    pred_class = torch.argmax(probs, dim=1).item()

    emotion = encoder.inverse_transform([pred_class])[0]
    confidence = probs[0][pred_class].item() * 100

    return emotion, round(confidence, 2)

In [27]:
def test_text(text):
    emotion, confidence = predict_text_emotion(text)
    print("Text:", text)
    print("Predicted:", emotion)
    print("Confidence:", confidence, "%")
    print()

test_text("I feel extremely happy today!")
test_text("I am very angry about what happened.")
test_text("I feel lonely and depressed.")
test_text("I am worry about my feature i think i should die.")

Text: I feel extremely happy today!
Predicted: happy
Confidence: 94.67 %

Text: I am very angry about what happened.
Predicted: angry
Confidence: 79.25 %

Text: I feel lonely and depressed.
Predicted: sad
Confidence: 93.12 %

Text: I am worry about my feature i think i should die.
Predicted: fear
Confidence: 59.0 %



In [28]:
# Save model
model.save_pretrained("text_emotion_model")

# Save tokenizer
tokenizer.save_pretrained("text_emotion_model")

# Save label encoder
import pickle
with open("text_label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Model saved successfully!")

Model saved successfully!


In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pickle

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("text_emotion_model")

# Load model
model = AutoModelForSequenceClassification.from_pretrained("text_emotion_model")

# Load encoder
with open("text_label_encoder.pkl", "rb") as f:
    encoder = pickle.load(f)

print("Model loaded successfully!")

W0225 15:40:44.849000 7056 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Model loaded successfully!


In [2]:
def predict_text_emotion(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=160
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    pred_class = torch.argmax(probs, dim=1).item()

    emotion = encoder.inverse_transform([pred_class])[0]
    confidence = probs[0][pred_class].item() * 100

    return emotion, round(confidence, 2)

In [3]:
predict_text_emotion("I very upset because i dint make my project")

('sad', 82.02)